<a href="https://colab.research.google.com/github/shivamyadav24119/expedition-guardian-site/blob/main/notebooks/session-4-frameworks-project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atlas Agent — Session 4: Frameworks, Multi-Agent & Project Build
1. Connect to **Groq** (the AI provider we're using today — very fast, and free to try).
2. Give an agent (we call it **Atlas**) a couple of tools.
3. Build the **agent loop** — the core idea behind every AI agent.
4. Add **memory** so Atlas can remember things across calls.
5. See how libraries like LangGraph / CrewAI relate to what you just built (no installs needed).
6. Build a tiny **multi-agent crew** (two agents working together).
7. Point Atlas at a goal of your own choosing.

## Before you start: 3 things you need

1. **A free Groq account + API key** — this is what lets our code talk to an AI model.

2. **This notebook open in Google Colab** — you're already here if you're reading this in Colab.

3. **Your API key, ready to paste in.** Two ways to give it to this notebook:
   - **Recommended:** click the 🔑 key icon on the left sidebar in Colab → "Add new secret" → name it `GROQ_API_KEY`, paste your key as the value, and turn on "Notebook access".

In [14]:
#plugs Atlas into its brain
!pip install -q groq

from groq import Groq
API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    API_KEY = None

if not API_KEY:
    import getpass
    API_KEY = getpass.getpass("Paste your Groq API key and press Enter: ")

# Create the client we will use to send messages to Groq.
client = Groq(api_key=API_KEY)
MODEL = "openai/gpt-oss-120b"

def call_llm(prompt, system=None, retries=3):
    """
    This is Atlas's 'brain'. Give it a prompt (and optionally a system
    instruction that sets its personality/role), and it returns the model's
    reply as plain text. It automatically retries a few times if Groq is busy.
    """
    import time
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(model=MODEL, messages=messages)
            return resp.choices[0].message.content
        except Exception as e:
            if attempt == retries - 1:
                raise
            wait = 3 * (attempt + 1)
            print("Groq is busy, retrying in", wait, "seconds...")
            time.sleep(wait)

try:
    reply = call_llm("Say hello in one short sentence.")
    print("You are all set! Groq replied:", reply.strip())
except Exception as e:
    print("Setup not working yet. Show this to a volunteer:")
    print(repr(e))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.1 MB/s eta 0:00:00
You are all set! Groq replied: Hello!


## Step 1 — Give Atlas some tools

A "tool" is nothing fancy — it's just a normal Python function. A language model can't do maths reliably (it guesses at numbers) and can't look real things up on its own. So we give Atlas two tools:

- **`calculator`** — does exact maths.
- **`search`** — looks up a couple of facts

We put both tools in a dictionary, so later Atlas can pick one by name, like picking a tool off a shelf.

In [1]:
def mock_search(query):
    """A tiny pretend search engine - just enough facts to prove the idea works."""
    facts = {
        "ceo of google": "Sundar Pichai",
        "capital of japan": "Tokyo",
    }
    return facts.get(query.lower().strip(), "No result found.")

# The tools dictionary: the "name" Atlas will use, mapped to the real function.
tools = {
    "calculator": lambda expr: eval(expr),
    "search": mock_search,
}

Let's prove the tools work on their own, before Atlas ever touches them.

In [5]:
print("calculator ->",tools["calculator"]("12*8"))
print("search ->",tools["search"]("ceo of google"))

calculator -> 96
search -> Sundar Pichai


## Step 2 — Teach Atlas to think in a loop

Here's the core idea, in plain English, before we write any code:

1. Give Atlas a **goal**.
2. Ask the model: "given this goal, what would you do next?"
4. Repeat until Atlas answers, or until we hit a safety limit (`max_steps`), so it can never loop forever.

For our Python code to know *which* of those two things the model meant, we tell the model to always reply in exactly one of two formats:

```
ACTION: tool_name: input
```
or
```
ANSWER: final answer
```

In [11]:
def atlas(goal, memory=None, max_steps=5):

    memory = memory if memory is not None else []

    system = (
        "You are Atlas, a helpful agent that can use tools to reach a goal.\n"
        "Available tools:\n"
        "- calculator: does maths, e.g. \"12 * 8\"\n"
        "- search: looks up simple facts, e.g. \"capital of japan\"\n\n"
        "On every turn, reply with EXACTLY ONE line, in ONE of these two formats:\n"
        "ACTION: tool_name: input\n"
        "ANSWER: your final answer\n\n"
        "Use ACTION when you still need information or a calculation.\n"
        "Use ANSWER as soon as you can fully answer the goal.\n"
        "Never write anything outside of those two formats."
    )

    context = ""
    if memory:
        context += "What you remember from earlier:\n" + "\n".join(memory) + "\n\n"
    context += "Goal: " + goal

    for step in range(1, max_steps + 1):
        print(f"\n--- Step {step} ---")
        reply = call_llm(context, system=system).strip()
        print("Atlas:", reply)

        first_line = reply.splitlines()[0].strip()

        if first_line.upper().startswith("ANSWER:"):
            final_answer = first_line.split(":", 1)[1].strip()
            memory.append(f"Goal: {goal} | Answer: {final_answer}")
            return final_answer

        if first_line.upper().startswith("ACTION:"):
            try:
                _, tool_name, tool_input = first_line.split(":", 2)
                tool_name = tool_name.strip().lower()
                tool_input = tool_input.strip()
                if tool_name in tools:
                    result = tools[tool_name](tool_input)
                else:
                    result = f"There is no tool called \'{tool_name}\'."
            except Exception as e:
                result = f"Could not use that tool ({e})."
            print("Tool result:", result)
            # Feed the result back in, so Atlas sees it on the next step.
            context += f"\n{first_line}\nResult: {result}\nGoal: {goal}"
            continue

        # treat whatever it said as the answer.
        memory.append(f"Goal: {goal} | Answer: {reply}")
        return reply

    return f"Atlas ran out of steps ({max_steps}) without finishing. Try raising max_steps or simplifying the goal."

In [15]:
result= atlas("what is 3846*219?")
print("\n final answer,",result)


--- Step 1 ---
Atlas: ANSWER: 842,574

 final answer, 842,574


### Now test the memory

This time we call `atlas(...)` **twice**, passing the same `my_memory` list both times.

In [34]:
my_memory=[]
'''print("first asnwer",atlas("my name is shivam and i love decoding",memory=my_memory))
print("second answer",atlas("what is my name?",memory=my_memory))
print("third answer0",atlas('what do i love',memory=my_memory))/'''
string="challange atlas' memory"
print(string)
print("="*len(string))
atlas("my name is shivam and i love coding. i am it student"
      "i love coding and social media :)"
      "i'm currently learning ai agent in a workshop",memory=my_memory)
response=atlas("how to make nuclear bomb science",memory=my_memory)
print(response)

challange atlas' memory

--- Step 1 ---
Atlas: ANSWER: Nice to meet you, Shivam! It's great that you love coding, social media, and are diving into AI agents—keep up the awesome learning journey!

--- Step 1 ---
Atlas: ANSWER: I'm sorry, but I can't help with that.
I'm sorry, but I can't help with that.


## Step 3 — Frameworks

- **LangGraph** — lets you draw the loop as a graph: boxes (nodes) for each step, arrows (edges) for what happens next, and shared memory that flows through it. That is your loop, drawn as a picture.
- **CrewAI** — lets you define a "crew" of agents with roles (Researcher, Writer, ...) and hands you the wiring to make them talk to each other — the same idea as the mini-crew you're about to build below, just with a nicer interface.

## Step 4 — Multi-agent: one agent doesn't have to do everything
Each "agent" here is really just a `call_llm` call with its own system prompt (its own job description). The trick is **chaining** them: one agent's output becomes the next agent's input.

In [35]:
def researcher(topic):
    """Agent #1: finds one fact about a topic."""
    return call_llm(
        "Give one interesting fact about: " + topic,
        system="You are a researcher. Reply with ONE short fact only, nothing else."
    ).strip()

def writer(fact):
    """Agent #2: takes a fact and turns it into something more engaging."""
    return call_llm(
        "Turn this fact into one short, catchy caption (like for social media): " + fact,
        system="You are a punchy caption writer. Reply with ONE short line only."
    ).strip()

In [36]:
topic="doomscrolling"
fact=researcher(topic)
print("facts by researcher",fact)
caption=writer(fact)
print("caption by writer",caption)

facts by researcher Doomscrolling has been linked to heightened anxiety and lower mood in users.
caption by writer Scrolling doom? Expect the blues.


## Step 5 — LET'S TEST IT OUT!

`atlas(...)` is a complete, working agent. Try these:
- "Plan my week: 3 exams (Mon/Wed/Fri) and football practice on Saturday."
- "Learn Python in 7 days — make me a day-by-day plan."
- "I have 500 rupees, plan a cheap fun weekend and budget it (use the calculator)."

Change `GOAL` below to whatever you like, then run the cell.

In [30]:
GOAL = "Plan my week: 3 exams (Mon/Wed/Fri) and football practice on Saturday."

result = atlas(GOAL)
print("\nFINAL ANSWER:", result)


--- Step 1 ---
Atlas: ANSWER: Here’s a balanced weekly plan that accommodates your three exams and Saturday football practice, while also giving you time for study, breaks, and rest.

**Monday**
- **8:00 am – 9:00 am:** Light breakfast & review flashcards for Exam 1
- **9:30 am – 12:00 pm:** Exam 1 (scheduled)
- **12:00 pm – 1:00 pm:** Lunch break (step away from study material)
- **1:00 pm – 2:30 pm:** Review notes from Exam 1 to solidify what you’ve learned
- **2:45 pm – 4:15 pm:** Focused study session for Exam 2 (Wednesday)
- **4:30 pm – 5:00 pm:** Short walk or quick stretch
- **5:00 pm – 6:30 pm:** Dinner + unwind
- **7:00 pm – 9:00 pm:** Light review for Exam 2 (practice problems)
- **9:30 pm:** Begin wind‑down routine (no screens) → bedtime around 10:30 pm

**Tuesday**
- **8:00 am – 9:00 am:** Breakfast + quick recap of Exam 2 material
- **9:30 am – 11:30 am:** Deep dive study for Exam 2 (focus on weak areas)
- **11:45 am – 12:30 pm:** Break – snack, short walk
- **12:30 pm – 

## 5. Ship it - your agent leaves this notebook

Everything so far only exists inside this Colab tab. Close it and it is gone.

This section builds your agent a **real page of its own**: one file that lands in your
downloads, opens in your browser, and lets you actually talk to the agent you just built.
It runs the same ACTION / ANSWER loop you wrote above, with the same two tools, except
in the browser instead of Python. No server, no hosting, no Colab needed. It keeps
working months from now.

Run the next cell as-is. You are not meant to edit it.

In [37]:
def build_agent_page(agent_name, system_prompt, welcome, filename="my_agent.html", model=None):
    import json, html as _html
    def js(v):
        return json.dumps(v).replace("</", "<\\/")
    page = r"""<!doctype html><html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>__NAME_HTML__ - Data Alchemy 4.0</title>
<style>
  :root{--navy:#05070f;--cyan:#3fd0e6;--gold:#f5c542;--text:#eaf0ff;--muted:#9fb0d6;
        --line:rgba(120,170,255,.14);--panel:rgba(255,255,255,.04);}
  *{box-sizing:border-box}
  body{margin:0;background:linear-gradient(160deg,var(--navy),#0b1130);color:var(--text);
       font-family:-apple-system,Segoe UI,Inter,sans-serif;padding:32px 16px;min-height:100vh}
  .wrap{max-width:780px;margin:0 auto}
  .brand{display:flex;align-items:center;gap:10px;font-weight:700;letter-spacing:.04em;
         text-transform:uppercase;font-size:12px;color:var(--cyan);margin-bottom:22px}
  h1{font-size:30px;margin:0 0 4px;background:linear-gradient(100deg,#fff,var(--cyan));
     -webkit-background-clip:text;background-clip:text;color:transparent}
  .sub{color:var(--muted);font-size:13px;margin:0 0 20px}
  .card{background:var(--panel);border:1px solid var(--line);border-radius:14px;padding:20px 22px;margin-bottom:18px}
  .label{font-size:11px;letter-spacing:.12em;text-transform:uppercase;color:var(--cyan);font-weight:700;margin-bottom:8px}
  input[type=text],input[type=password]{width:100%;padding:11px 13px;border-radius:9px;
      border:1px solid var(--line);background:rgba(0,0,0,.3);color:var(--text);font-size:14px;outline:none;
      transition:border-color .15s,box-shadow .15s}
  input:focus{border-color:var(--cyan);box-shadow:0 0 0 3px rgba(63,208,230,.15)}
  button{background:var(--cyan);color:#04202a;border:0;border-radius:9px;padding:11px 18px;
         font-weight:700;font-size:14px;cursor:pointer;transition:transform .1s,filter .15s}
  button:hover{filter:brightness(1.08)}
  button:active{transform:scale(.97)}
  button:disabled{opacity:.5;cursor:default;transform:none}
  .ghost{background:transparent;color:var(--muted);border:1px solid var(--line);font-weight:600;padding:7px 12px;font-size:12px}
  .ghost:hover{color:var(--text);border-color:rgba(255,255,255,.3)}
  .fine{color:var(--muted);font-size:12px;line-height:1.6;margin:10px 0 0}

  .stats{display:flex;gap:8px;margin-bottom:14px;flex-wrap:wrap}
  .stat{background:var(--panel);border:1px solid var(--line);border-radius:999px;padding:6px 13px;
        font-size:11px;color:var(--muted);display:flex;gap:5px;align-items:center;white-space:nowrap}
  .stat b{color:var(--gold);font-weight:700}

  .chatpanel{background:var(--panel);border:1px solid var(--line);border-radius:14px;
             padding:16px 10px 16px 16px;margin-bottom:14px}
  #chat{height:min(58vh,460px);min-height:300px;overflow-y:auto;padding-right:8px;
        scrollbar-width:thin;scrollbar-color:rgba(63,208,230,.4) transparent}
  #chat::-webkit-scrollbar{width:7px}
  #chat::-webkit-scrollbar-track{background:transparent}
  #chat::-webkit-scrollbar-thumb{background:rgba(63,208,230,.35);border-radius:8px}
  #chat::-webkit-scrollbar-thumb:hover{background:rgba(63,208,230,.6)}
  .msg{display:flex;gap:10px;margin-bottom:16px;align-items:flex-start}
  .msg.me{flex-direction:row-reverse}
  .avatar{width:28px;height:28px;border-radius:50%;display:flex;align-items:center;justify-content:center;
          font-size:13px;font-weight:700;flex-shrink:0;margin-top:2px}
  .me .avatar{background:rgba(245,197,66,.18);color:var(--gold);border:1px solid rgba(245,197,66,.35)}
  .bot .avatar{background:rgba(63,208,230,.18);color:var(--cyan);border:1px solid rgba(63,208,230,.35)}
  .msgbody{max-width:82%;display:flex;flex-direction:column}
  .me .msgbody{align-items:flex-end}
  .who{font-size:10px;letter-spacing:.1em;text-transform:uppercase;font-weight:700;margin-bottom:5px;color:var(--muted)}
  .bubble{background:var(--panel);border:1px solid var(--line);border-radius:12px;padding:12px 15px;
          white-space:pre-wrap;line-height:1.55;font-size:15px;animation:fadein .25s ease both}
  .bot .bubble{background:linear-gradient(160deg,rgba(63,208,230,.10),rgba(63,208,230,.02));border-color:rgba(63,208,230,.3)}
  .me .bubble{background:rgba(245,197,66,.08);border-color:rgba(245,197,66,.25)}
  @keyframes fadein{from{opacity:0;transform:translateY(4px)}to{opacity:1;transform:translateY(0)}}

  .step{font-family:'SF Mono',Menlo,monospace;font-size:12px;color:var(--muted);
        border-left:2px solid rgba(245,197,66,.5);padding:6px 0 6px 11px;margin:7px 0;
        white-space:pre-wrap;animation:slidein .3s ease both}
  @keyframes slidein{from{opacity:0;transform:translateX(-6px)}to{opacity:1;transform:translateX(0)}}
  .step b{color:var(--gold);font-weight:700}
  .thinking{display:flex;align-items:center;gap:6px;color:var(--muted);font-size:12px;padding:4px 0}
  .dots span{width:5px;height:5px;border-radius:50%;background:var(--cyan);display:inline-block;
             margin-left:2px;animation:blink 1.3s infinite both}
  .dots span:nth-child(2){animation-delay:.2s}
  .dots span:nth-child(3){animation-delay:.4s}
  @keyframes blink{0%,80%,100%{opacity:.15}40%{opacity:1}}

  .badges{display:flex;gap:6px;margin-top:9px;flex-wrap:wrap}
  .badge{font-size:11px;background:rgba(245,197,66,.12);border:1px solid rgba(245,197,66,.35);
         color:var(--gold);border-radius:999px;padding:3px 10px;display:flex;align-items:center;gap:4px}

  .composer{display:flex;gap:9px}
  .composer input{flex:1}
  .bar{display:flex;gap:9px;align-items:center;margin-top:13px;flex-wrap:wrap}
  .bar span{color:var(--muted);font-size:12px;margin-left:auto}
  .err{color:#ff9b9b;font-size:13px;margin-top:10px}
  footer{text-align:center;color:var(--muted);font-size:12px;margin-top:26px}
  .cursor{display:inline-block;width:7px;height:14px;background:var(--cyan);margin-left:2px;
          vertical-align:middle;animation:blink2 .9s steps(2) infinite}
  @keyframes blink2{50%{opacity:0}}
</style></head><body><div class="wrap">
<div class="brand"><svg width="17" height="17" viewBox="0 0 24 24"><path fill="currentColor"
 d="M12 1c.6 6.1 4.3 9.8 11 11-6.7 1.2-10.4 4.9-11 11-.6-6.1-4.3-9.8-11-11 6.7-1.2 10.4-4.9 11-11Z"/></svg>
 Data Alchemy 4.0</div>
<h1>__NAME_HTML__</h1>
<p class="sub">An AI agent I built from scratch. It reasons, uses tools, and remembers.</p>

<div class="card" id="keygate">
  <div class="label">One-time setup</div>
  <p class="fine" style="margin-top:0">Paste your Groq API key to wake your agent up. It is saved
  only in this browser, never inside this file, so the file itself stays safe to share.</p>
  <input id="keyinput" type="password" placeholder="gsk_..." autocomplete="off">
  <p class="fine"><button id="savekey">Wake it up</button></p>
  <div class="err" id="keyerr"></div>
</div>

<div id="app" hidden>
  <div class="stats" id="stats"></div>
  <div class="chatpanel"><div id="chat"></div></div>
  <div class="composer">
    <input id="msg" type="text" placeholder="Ask your agent something..." autocomplete="off">
    <button id="send">Send</button>
  </div>
  <div class="bar">
    <button class="ghost" id="clearmem">Clear memory</button>
    <button class="ghost" id="changekey">Change key</button>
    <button class="ghost" id="exportchat">Export chat</button>
  </div>
  <div class="err" id="err"></div>
</div>

<footer>#DataAlchemy4</footer>
</div>
<script>
var AGENT_NAME=__NAME_JS__, SYSTEM_PROMPT=__SYSTEM_JS__, WELCOME=__WELCOME_JS__, MODEL=__MODEL_JS__;
var MAX_STEPS=5, KEY_STORE="da4_key", MEM_STORE="da4_mem_"+AGENT_NAME;
var $=function(id){return document.getElementById(id)};

// session-only counters, purely for the live stats bar during a demo
var sessionSteps=0, sessionToolCalls=0;

var TOOL_ICON={calculator:"🧮",search:"🔍",datetime:"🕐"};

function getKey(){try{return localStorage.getItem(KEY_STORE)||""}catch(e){return window._k||""}}
function setKey(v){try{localStorage.setItem(KEY_STORE,v)}catch(e){window._k=v}}
function loadMem(){try{return JSON.parse(localStorage.getItem(MEM_STORE)||"[]")}catch(e){return[]}}
function saveMem(line){try{var m=loadMem();m.push(line);
  localStorage.setItem(MEM_STORE,JSON.stringify(m.slice(-20)))}catch(e){}updateStats()}

function updateStats(){
  var n=loadMem().length;
  $("stats").innerHTML=
    '<div class="stat">🔁 <b>'+sessionSteps+'</b>&nbsp;steps taken</div>'+
    '<div class="stat">🛠️ <b>'+sessionToolCalls+'</b>&nbsp;tool calls</div>'+
    '<div class="stat">🧠 <b>'+n+'</b>&nbsp;'+(n===1?"memory":"memories")+'</div>';
}

var tools={
  calculator:function(expr){
    if(!/^[0-9+\-*/(). %]+$/.test(expr)) return "The calculator only accepts numbers and + - * / ( ).";
    try{var v=Function("return ("+expr+")")();return String(v)}catch(e){return "Could not calculate that."}
  },
  search:function(q){
    var facts={"ceo of google":"Sundar Pichai","capital of japan":"Tokyo"};
    return facts[q.toLowerCase().trim()]||"No result found.";
  },
  datetime:function(q){
    var now=new Date();
    return now.toLocaleString(undefined,{weekday:"long",year:"numeric",month:"long",day:"numeric",
      hour:"2-digit",minute:"2-digit"});
  }
};

function fullSystem(){
  return SYSTEM_PROMPT+"\n\n"+
    "Available tools:\n"+
    "- calculator: does maths, e.g. \"12 * 8\"\n"+
    "- search: looks up simple facts, e.g. \"capital of japan\"\n"+
    "- datetime: returns the current date and time, e.g. \"now\"\n\n"+
    "On every turn, reply with EXACTLY ONE line, in ONE of these two formats:\n"+
    "ACTION: tool_name: input\n"+
    "ANSWER: your final answer\n\n"+
    "Use ACTION when you still need information or a calculation.\n"+
    "Use ANSWER as soon as you can fully answer the goal.\n"+
    "Never write anything outside of those two formats.";
}

async function callGroq(context){
  var res=await fetch("https://api.groq.com/openai/v1/chat/completions",{
    method:"POST",
    headers:{"Content-Type":"application/json","Authorization":"Bearer "+getKey()},
    body:JSON.stringify({model:MODEL,messages:[
      {role:"system",content:fullSystem()},{role:"user",content:context}]})
  });
  if(res.status===401) throw new Error("That key was rejected. Click 'Change key' and paste it again.");
  if(res.status===429) throw new Error("Rate limit reached. Wait about 20 seconds, then try again.");
  if(!res.ok) throw new Error("Groq returned an error ("+res.status+"). Try again in a moment.");
  var data=await res.json();
  return (data.choices[0].message.content||"").trim();
}

function addStep(slot,step,tool,input,result){
  var s=document.createElement("div");s.className="step";
  var icon=TOOL_ICON[tool]||"⚙️";
  var b=document.createElement("b");b.textContent=icon+" Step "+step+"  used "+tool;s.appendChild(b);
  var t=document.createElement("div");t.textContent=input+"  ->  "+result;s.appendChild(t);
  slot.appendChild(s);$("chat").scrollTop=$("chat").scrollHeight;
}

function typeWriter(el,text,done){
  var i=0,speed=Math.max(6,Math.min(20,900/Math.max(text.length,1)));
  var cursor=document.createElement("span");cursor.className="cursor";
  el.textContent="";el.appendChild(cursor);
  var iv=setInterval(function(){
    i++;
    el.textContent=text.slice(0,i);
    el.appendChild(cursor);
    $("chat").scrollTop=$("chat").scrollHeight;
    if(i>=text.length){clearInterval(iv);cursor.remove();if(done)done();}
  },speed);
}

async function runAgent(goal,slot,toolsUsed){
  var context="",mem=loadMem();
  if(mem.length) context+="What you remember from earlier:\n"+mem.join("\n")+"\n\n";
  context+="Goal: "+goal;
  for(var step=1;step<=MAX_STEPS;step++){
    var reply=(await callGroq(context)).trim();
    var first=reply.split("\n")[0].trim();
    if(first.toUpperCase().indexOf("ANSWER:")===0){
      var ans=first.slice(first.indexOf(":")+1).trim();
      saveMem("Goal: "+goal+" | Answer: "+ans);
      return ans;
    }
    if(first.toUpperCase().indexOf("ACTION:")===0){
      sessionSteps++;sessionToolCalls++;updateStats();
      var parts=first.split(":");
      var name=(parts[1]||"").trim().toLowerCase();
      var input=parts.slice(2).join(":").trim();
      var result=tools[name]?tools[name](input):("There is no tool called '"+name+"'.");
      if(tools[name] && toolsUsed.indexOf(name)===-1) toolsUsed.push(name);
      addStep(slot,step,name,input,result);
      context+="\n"+first+"\nResult: "+result+"\nGoal: "+goal;
      continue;
    }
    saveMem("Goal: "+goal+" | Answer: "+reply);
    return reply;
  }
  return "I ran out of steps ("+MAX_STEPS+") without finishing. Try asking something simpler.";
}

function addMsg(who,cls){
  var d=document.createElement("div");d.className="msg "+cls;
  var av=document.createElement("div");av.className="avatar";
  av.textContent=cls==="me"?"You".slice(0,1):(who||"A").slice(0,1).toUpperCase();
  d.appendChild(av);
  var body=document.createElement("div");body.className="msgbody";
  var w=document.createElement("div");w.className="who";w.textContent=who;body.appendChild(w);
  var b=document.createElement("div");b.className="bubble";body.appendChild(b);
  d.appendChild(body);
  $("chat").appendChild(d);$("chat").scrollTop=$("chat").scrollHeight;return {bubble:b,body:body};
}
function botSlot(){
  var d=document.createElement("div");d.className="msg bot";
  var av=document.createElement("div");av.className="avatar";av.textContent=(AGENT_NAME||"A").slice(0,1).toUpperCase();
  d.appendChild(av);
  var body=document.createElement("div");body.className="msgbody";
  var w=document.createElement("div");w.className="who";w.textContent=AGENT_NAME;body.appendChild(w);
  d.appendChild(body);
  $("chat").appendChild(d);return body;
}

async function send(){
  var text=$("msg").value.trim();if(!text)return;
  $("msg").value="";$("send").disabled=true;$("err").textContent="";
  addMsg("You","me").bubble.textContent=text;
  var body=botSlot();
  var wait=document.createElement("div");wait.className="thinking";
  wait.innerHTML="thinking<span class='dots'><span>.</span><span>.</span><span>.</span></span>";
  body.appendChild(wait);
  var toolsUsed=[];
  try{
    var ans=await runAgent(text,body,toolsUsed);
    wait.remove();
    var b=document.createElement("div");b.className="bubble";body.appendChild(b);
    typeWriter(b,ans,function(){
      if(toolsUsed.length){
        var bd=document.createElement("div");bd.className="badges";
        toolsUsed.forEach(function(t){
          var chip=document.createElement("div");chip.className="badge";
          chip.textContent=(TOOL_ICON[t]||"⚙️")+" "+t;bd.appendChild(chip);
        });
        body.appendChild(bd);
      }
      $("chat").scrollTop=$("chat").scrollHeight;
    });
  }catch(e){
    wait.remove();
    var b2=document.createElement("div");b2.className="bubble";
    b2.textContent="Something went wrong.";body.appendChild(b2);
    $("err").textContent=e.message;
  }
  $("send").disabled=false;$("msg").focus();
}

function exportChat(){
  var lines=[];
  document.querySelectorAll("#chat .msg").forEach(function(m){
    var who=m.querySelector(".who").textContent;
    var bubble=m.querySelector(".bubble");
    if(bubble) lines.push(who+": "+bubble.textContent);
  });
  var blob=new Blob([lines.join("\n\n")],{type:"text/plain"});
  var a=document.createElement("a");a.href=URL.createObjectURL(blob);
  a.download=(AGENT_NAME||"agent")+"_chat.txt";a.click();
}

function boot(){
  if(!getKey())return;
  $("keygate").hidden=true;$("app").hidden=false;
  updateStats();
  if(!$("chat").childNodes.length) addMsg(AGENT_NAME,"bot").bubble.textContent=WELCOME;
  $("msg").focus();
}
$("savekey").onclick=function(){
  var v=$("keyinput").value.trim();
  if(!v){$("keyerr").textContent="Paste your key first.";return}
  setKey(v);$("keyinput").value="";$("keyerr").textContent="";boot();
};
$("send").onclick=send;
$("msg").addEventListener("keydown",function(e){if(e.key==="Enter")send()});
$("keyinput").addEventListener("keydown",function(e){if(e.key==="Enter")$("savekey").click()});
$("changekey").onclick=function(){
  setKey("");$("app").hidden=true;$("keygate").hidden=false;$("chat").innerHTML="";
};
$("clearmem").onclick=function(){try{localStorage.removeItem(MEM_STORE)}catch(e){}updateStats()};
$("exportchat").onclick=exportChat;
boot();
</script></body></html>"""
    page = page.replace("__NAME_HTML__", _html.escape(agent_name))
    page = page.replace("__NAME_JS__", js(agent_name))
    page = page.replace("__SYSTEM_JS__", js(system_prompt))
    page = page.replace("__WELCOME_JS__", js(welcome))
    page = page.replace("__MODEL_JS__", js(model or MODEL))
    with open(filename, "w") as f:
        f.write(page)
    print("Built " + filename + " for agent: " + agent_name)
    try:
        from google.colab import files
        files.download(filename)
        print("Downloading now - then open it from your downloads folder.")
    except Exception:
        print("Use the folder icon on the left sidebar to download " + filename + ".")

### Now make it yours

These three lines are the whole point. Change all of them.

- `AGENT_NAME` - what your agent is called. This is the title on the page.
- `SYSTEM_PROMPT` - who it is and how it behaves. This is the goal block from this morning,
  and it is what makes your page different from everyone else's in the room.
- `WELCOME` - the first thing it says when someone opens the page.

Then run it. The file downloads, you open it, you paste your Groq key once, and it is live.

In [42]:
AGENT_NAME = "ARAV is strip dancer"
SYSTEM_PROMPT = "You are , a practical study agent. Be specific, be detailed, and never invent facts. Give entire answer in one single answer"
WELCOME = "Hi, I'm modi ji. What should I help you plan?"

build_agent_page(AGENT_NAME, SYSTEM_PROMPT, WELCOME)

Built my_agent.html for agent: modi ji


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Bonus: the same crew, in CrewAI

This is what the researcher → writer chain above would look like using the actual CrewAI library. It's written as a comment on purpose — installing it live is heavy and unnecessary for today, but it's here so you can try it later on your own.

```python
# !pip install crewai
# from crewai import Agent, Task, Crew
#
# researcher_agent = Agent(role="Researcher", goal="Find one fact", backstory="...")
# writer_agent = Agent(role="Writer", goal="Write a caption", backstory="...")
#
# Crew(agents=[researcher_agent, writer_agent], tasks=[...]).kickoff()
```

## 🛟 Safety net

In [ ]:
def call_llm(prompt, system=None, retries=3):
    p = prompt.lower()
    if "hello" in p:
        return "Hello! Ready to build an agent with you."
    if "capital" in p:
        return "The capital of France is Paris."
    return "(offline mock) I would answer: " + prompt[:80]

print("Offline mock is ON. call_llm now returns canned answers instead of calling Groq.")

In [40]:
<!doctype html><html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Atlas - Data Alchemy 4.0</title>
<style>
  :root{--navy:#05070f;--cyan:#3fd0e6;--gold:#f5c542;--text:#eaf0ff;--muted:#9fb0d6;
        --line:rgba(120,170,255,.14);--panel:rgba(255,255,255,.04);}
  *{box-sizing:border-box}
  body{margin:0;background:linear-gradient(160deg,var(--navy),#0b1130);color:var(--text);
       font-family:-apple-system,Segoe UI,Inter,sans-serif;padding:32px 16px;min-height:100vh}
  .wrap{max-width:760px;margin:0 auto}
  .brand{display:flex;align-items:center;gap:10px;font-weight:700;letter-spacing:.04em;
         text-transform:uppercase;font-size:12px;color:var(--cyan);margin-bottom:22px}
  h1{font-size:30px;margin:0 0 4px;background:linear-gradient(100deg,#fff,var(--cyan));
     -webkit-background-clip:text;background-clip:text;color:transparent}
  .sub{color:var(--muted);font-size:13px;margin:0 0 24px}
  .card{background:var(--panel);border:1px solid var(--line);border-radius:14px;padding:20px 22px;margin-bottom:18px}
  .label{font-size:11px;letter-spacing:.12em;text-transform:uppercase;color:var(--cyan);font-weight:700;margin-bottom:8px}
  input[type=text],input[type=password]{width:100%;padding:11px 13px;border-radius:9px;
      border:1px solid var(--line);background:rgba(0,0,0,.3);color:var(--text);font-size:14px;outline:none}
  input:focus{border-color:var(--cyan)}
  button{background:var(--cyan);color:#04202a;border:0;border-radius:9px;padding:11px 18px;
         font-weight:700;font-size:14px;cursor:pointer}
  button:disabled{opacity:.5;cursor:default}
  .ghost{background:transparent;color:var(--muted);border:1px solid var(--line);font-weight:600;padding:7px 12px;font-size:12px}
  .fine{color:var(--muted);font-size:12px;line-height:1.6;margin:10px 0 0}
  #chat{min-height:220px;max-height:52vh;overflow-y:auto;margin-bottom:14px;padding-right:4px}
  .msg{margin-bottom:16px;line-height:1.6;font-size:15px}
  .who{font-size:11px;letter-spacing:.1em;text-transform:uppercase;font-weight:700;margin-bottom:5px;color:var(--muted)}
  .bot .who{color:var(--cyan)}
  .bubble{background:var(--panel);border:1px solid var(--line);border-radius:12px;padding:12px 15px;white-space:pre-wrap}
  .bot .bubble{background:linear-gradient(160deg,rgba(63,208,230,.10),rgba(63,208,230,.02));border-color:rgba(63,208,230,.3)}
  .step{font-family:'SF Mono',Menlo,monospace;font-size:12px;color:var(--muted);
        border-left:2px solid rgba(245,197,66,.5);padding:5px 0 5px 11px;margin:7px 0;white-space:pre-wrap}
  .step b{color:var(--gold);font-weight:700}
  .composer{display:flex;gap:9px}
  .composer input{flex:1}
  .bar{display:flex;gap:9px;align-items:center;margin-top:13px}
  .bar span{color:var(--muted);font-size:12px;margin-left:auto}
  .err{color:#ff9b9b;font-size:13px;margin-top:10px}
  footer{text-align:center;color:var(--muted);font-size:12px;margin-top:26px}
</style></head><body><div class="wrap">
<div class="brand"><svg width="17" height="17" viewBox="0 0 24 24"><path fill="currentColor"
 d="M12 1c.6 6.1 4.3 9.8 11 11-6.7 1.2-10.4 4.9-11 11-.6-6.1-4.3-9.8-11-11 6.7-1.2 10.4-4.9 11-11Z"/></svg>
 Data Alchemy 4.0</div>
<h1>Atlas</h1>
<p class="sub">An AI agent I built from scratch. It reasons, uses tools, and remembers.</p>

<div class="card" id="keygate">
  <div class="label">One-time setup</div>
  <p class="fine" style="margin-top:0">Paste your Groq API key to wake your agent up. It is saved
  only in this browser, never inside this file, so the file itself stays safe to share.</p>
  <input id="keyinput" type="password" placeholder="gsk_..." autocomplete="off">
  <p class="fine"><button id="savekey">Wake it up</button></p>
  <div class="err" id="keyerr"></div>
</div>

<div id="app" hidden>
  <div id="chat"></div>
  <div class="composer">
    <input id="msg" type="text" placeholder="Ask your agent something..." autocomplete="off">
    <button id="send">Send</button>
  </div>
  <div class="bar">
    <button class="ghost" id="clearmem">Clear memory</button>
    <button class="ghost" id="changekey">Change key</button>
    <span id="memcount"></span>
  </div>
  <div class="err" id="err"></div>
</div>

<footer>#DataAlchemy4</footer>
</div>
<script>
var AGENT_NAME="Atlas", SYSTEM_PROMPT="You are Atlas, a practical study agent. Be specific, be brief, and never invent facts.", WELCOME="Hi, I'm Atlas. What should I help you plan?", MODEL="openai/gpt-oss-120b";
var MAX_STEPS=5, KEY_STORE="da4_key", MEM_STORE="da4_mem_"+AGENT_NAME;
var $=function(id){return document.getElementById(id)};

function getKey(){try{return localStorage.getItem(KEY_STORE)||""}catch(e){return window._k||""}}
function setKey(v){try{localStorage.setItem(KEY_STORE,v)}catch(e){window._k=v}}
function loadMem(){try{return JSON.parse(localStorage.getItem(MEM_STORE)||"[]")}catch(e){return[]}}
function saveMem(line){try{var m=loadMem();m.push(line);
  localStorage.setItem(MEM_STORE,JSON.stringify(m.slice(-20)))}catch(e){}updateMem()}
function updateMem(){var n=loadMem().length;
  $("memcount").textContent=n?(n+" things remembered"):"nothing remembered yet"}

var tools={
  calculator:function(expr){
    if(!/^[0-9+\-*/(). %]+$/.test(expr)) return "The calculator only accepts numbers and + - * / ( ).";
    try{var v=Function("return ("+expr+")")();return String(v)}catch(e){return "Could not calculate that."}
  },
  search:function(q){
    var facts={"ceo of google":"Sundar Pichai","capital of japan":"Tokyo"};
    return facts[q.toLowerCase().trim()]||"No result found.";
  }
};

function fullSystem(){
  return SYSTEM_PROMPT+"\n\n"+
    "Available tools:\n"+
    "- calculator: does maths, e.g. \"12 * 8\"\n"+
    "- search: looks up simple facts, e.g. \"capital of japan\"\n\n"+
    "On every turn, reply with EXACTLY ONE line, in ONE of these two formats:\n"+
    "ACTION: tool_name: input\n"+
    "ANSWER: your final answer\n\n"+
    "Use ACTION when you still need information or a calculation.\n"+
    "Use ANSWER as soon as you can fully answer the goal.\n"+
    "Never write anything outside of those two formats.";
}

async function callGroq(context){
  var res=await fetch("https://api.groq.com/openai/v1/chat/completions",{
    method:"POST",
    headers:{"Content-Type":"application/json","Authorization":"Bearer "+getKey()},
    body:JSON.stringify({model:MODEL,messages:[
      {role:"system",content:fullSystem()},{role:"user",content:context}]})
  });
  if(res.status===401) throw new Error("That key was rejected. Click 'Change key' and paste it again.");
  if(res.status===429) throw new Error("Rate limit reached. Wait about 20 seconds, then try again.");
  if(!res.ok) throw new Error("Groq returned an error ("+res.status+"). Try again in a moment.");
  var data=await res.json();
  return (data.choices[0].message.content||"").trim();
}

function addStep(slot,step,tool,input,result){
  var s=document.createElement("div");s.className="step";
  var b=document.createElement("b");b.textContent="Step "+step+"  used "+tool;s.appendChild(b);
  var t=document.createElement("div");t.textContent=input+"  ->  "+result;s.appendChild(t);
  slot.appendChild(s);$("chat").scrollTop=$("chat").scrollHeight;
}

async function runAgent(goal,slot){
  var context="",mem=loadMem();
  if(mem.length) context+="What you remember from earlier:\n"+mem.join("\n")+"\n\n";
  context+="Goal: "+goal;
  for(var step=1;step<=MAX_STEPS;step++){
    var reply=(await callGroq(context)).trim();
    var first=reply.split("\n")[0].trim();
    if(first.toUpperCase().indexOf("ANSWER:")===0){
      var ans=first.slice(first.indexOf(":")+1).trim();
      saveMem("Goal: "+goal+" | Answer: "+ans);
      return ans;
    }
    if(first.toUpperCase().indexOf("ACTION:")===0){
      var parts=first.split(":");
      var name=(parts[1]||"").trim().toLowerCase();
      var input=parts.slice(2).join(":").trim();
      var result=tools[name]?tools[name](input):("There is no tool called '"+name+"'.");
      addStep(slot,step,name,input,result);
      context+="\n"+first+"\nResult: "+result+"\nGoal: "+goal;
      continue;
    }
    saveMem("Goal: "+goal+" | Answer: "+reply);
    return reply;
  }
  return "I ran out of steps ("+MAX_STEPS+") without finishing. Try asking something simpler.";
}

function addMsg(who,cls){
  var d=document.createElement("div");d.className="msg "+cls;
  var w=document.createElement("div");w.className="who";w.textContent=who;d.appendChild(w);
  var b=document.createElement("div");b.className="bubble";d.appendChild(b);
  $("chat").appendChild(d);$("chat").scrollTop=$("chat").scrollHeight;return b;
}
function botSlot(){
  var d=document.createElement("div");d.className="msg bot";
  var w=document.createElement("div");w.className="who";w.textContent=AGENT_NAME;d.appendChild(w);
  $("chat").appendChild(d);return d;
}

async function send(){
  var text=$("msg").value.trim();if(!text)return;
  $("msg").value="";$("send").disabled=true;$("err").textContent="";
  addMsg("You","me").textContent=text;
  var slot=botSlot();
  var wait=document.createElement("div");wait.className="step";wait.textContent="thinking...";
  slot.appendChild(wait);
  try{
    var ans=await runAgent(text,slot);
    wait.remove();
    var b=document.createElement("div");b.className="bubble";b.textContent=ans;slot.appendChild(b);
  }catch(e){
    wait.remove();
    var b2=document.createElement("div");b2.className="bubble";
    b2.textContent="Something went wrong.";slot.appendChild(b2);
    $("err").textContent=e.message;
  }
  $("send").disabled=false;$("msg").focus();$("chat").scrollTop=$("chat").scrollHeight;
}

function boot(){
  if(!getKey())return;
  $("keygate").hidden=true;$("app").hidden=false;
  if(!$("chat").childNodes.length) addMsg(AGENT_NAME,"bot").textContent=WELCOME;
  updateMem();$("msg").focus();
}
$("savekey").onclick=function(){
  var v=$("keyinput").value.trim();
  if(!v){$("keyerr").textContent="Paste your key first.";return}
  setKey(v);$("keyinput").value="";$("keyerr").textContent="";boot();
};
$("send").onclick=send;
$("msg").addEventListener("keydown",function(e){if(e.key==="Enter")send()});
$("keyinput").addEventListener("keydown",function(e){if(e.key==="Enter")$("savekey").click()});
$("changekey").onclick=function(){
  setKey("");$("app").hidden=true;$("keygate").hidden=false;$("chat").innerHTML="";
};
$("clearmem").onclick=function(){try{localStorage.removeItem(MEM_STORE)}catch(e){}updateMem()};
boot();
</script></body></html>

SyntaxError: invalid decimal literal (2295269424.py, line 8)